### Big Data com duckdb Local

In [ ]:
## Passos para montagem de um Mini Sistema de analise de BIG Data Local Com duckdb

## carregar ficheiro para o banco de dados duckdb (staging Row Data)
## Certificar que o ficheiro carregado uma vez nao seja mas carregado 
## Tranformar dados antes de passar para camada (dados limpos)
## Salvar ficheiro em parquet uso no excel e Power BI
## salvar dados banco de dados fisico duckdb (tabelas e viwes analise rapido)
## salvar logs de transacao de dados 

In [60]:
# bibliotecas necessarias 

import os
import duckdb
from datetime import date
import pandas as pd
import polars as pl
from sqlalchemy import create_engine

In [ ]:
## Criando conexao com banco de dados duckdb

con = duckdb.connect("..//db//relatorio_vendas_telemoveis.duckdb")

#### Criando Tabela Ficheiro Processamento e Registro Logs

In [57]:
## Criando Tabelas de logs de dados 

# Criando tabela de ficheiro vendas ja processados 

con.execute(
    """CREATE TABLE IF NOT EXISTS ficheiro_processados
    (
        nome_ficheiro varchar(50),
        data_precessado date default now()
    );
    """
)

con.execute(
    """
    CREATE TABLE IF NOT EXISTS log_carga 
    (
    nome_ficheiro varchar(50),
    data_evento   date,
    status        varchar(50),      -- 'SUCESSO' ou 'ERRO'
    mensagem      varchar(50)
);
""")

#### Cargas de Dados Brutos Ficheiro CSV -> Tabela Temp duckdb

In [ ]:
## CARREGAR DADOS PARA CAMADA DE DADOS BRUTOS

""" Atraves do OS podemos pegar o caminho da pasta e com loop for 
    carregar cada ficheiro e tipo carga de dados da pasta como faz power query
    por enquanto realizo carga manual dos dados

    Melhoria encpsular isso dentro de uma funcao 
"""
# ficheiro da kaggle na pasta link

caminho = "C:\\Users\\Jeremias\\Desktop\\Outros Projectos\\Base de Dados\\BIG DATA SALES\\2019-Oct.csv"
nome_ficheiro = os.path.basename(caminho)

## verifica se esse ficheiro ja foi processado
verificar = con.execute(f"select 1 from ficheiro_processados where nome_ficheiro = '{nome_ficheiro}'").fetchone()

if verificar == None:

    # Apaga a tabela de staging caso exista.
    # Os dados brutos são temporários e usados apenas para processamento.
    con.execute("DROP TABLE IF EXISTS vendas_brutos")

    # Cria uma tabela temporária (staging) com a estrutura por inferencia do csv
    # e carrega os dados para uso na camada de processamento dados brutos
    # podia usar CREATE OR REPLACE TEMP TABLE vendas_brutos AS evitaria uso do DROP

    try:
        con.execute("""
            CREATE TEMP TABLE vendas_brutos AS
                SELECT * FROM read_csv_auto(?)
        """, (caminho,))

        # inserir nome do ficheiro processado na tabela de ficheiro processados
        con.execute("INSERT INTO ficheiro_processados (nome_ficheiro) VALUES (?) ", (nome_ficheiro,))

        # salvando registro de log de sucesso
        con.execute("""
        INSERT INTO log_carga (nome_ficheiro, data_evento, status, mensagem)
        VALUES (?, ?, ?, ?)
        """, (
        nome_ficheiro,
        date.today(),
        "SUCESSO",
        "Ficheiro carregado com sucesso"
        ))

    except Exception as e:
        
        # salvando registro de log de falha
        con.execute("""
        INSERT INTO log_carga (nome_ficheiro, data_evento, status, mensagem)
        VALUES (?, ?, ?, ?)
        """, (
        nome_ficheiro,
        date.today(),
        "ERRO",
        str(e)
        ))

    print("sucesso na carga dos dados")

else:

    ## Enviar mensagem no email substitui print
    print(f'ficheiro {nome_ficheiro} ja foi processado')


ficheiro 2019-Oct.csv ja foi processado


#### Carga de dados tabela dados brutos para tabela limpas

In [74]:
# carga de dados para camada de dados limpos e atualizacao incremental

tabela_existe = con.execute("""SELECT COUNT(*) FROM information_schema.tables WHERE table_name='vendas_limpas'
""").fetchone()[0] == 1

if not tabela_existe:
    con.execute("""
        CREATE TABLE vendas_limpas AS
        SELECT
            CAST(event_time AS DATE) AS Tempo_evento,
            COALESCE(event_type , 'Sem Tipo Evento') AS Tipo_evento,
            product_id AS Codigo_produto,
            category_id AS Codigo_categoria,
            COALESCE(brand , 'Sem marcas') AS Marca,
            price AS preco,
            user_id AS Codigo_cliente
        FROM vendas_brutos
    """)
else:
    con.execute("""
        INSERT INTO vendas_limpas
        SELECT
            CAST(event_time AS DATE) AS Tempo_evento,
            COALESCE(event_type , 'Sem Tipo Evento') AS Tipo_evento,
            product_id AS Codigo_produto,
            category_id AS Codigo_categoria,
            COALESCE(brand , 'Sem marcas') AS Marca,
            price AS preco,
            user_id AS Codigo_cliente
        FROM vendas_brutos
        WHERE CAST(event_time AS DATE) > (SELECT COALESCE(MAX(Tempo_evento), DATE '1900-01-01') FROM vendas_limpas)
    """)

print('Dados carregados para Tabelas dados limpos')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dados carregados para Tabelas dados limpos


#### Teste dados Dados

In [ ]:
# conprovacao que os dados passou da tabela brutas para limpas

con.execute(""" select * from vendas_limpas limit 5""").fetch_df()


,Tempo_evento,Tipo_evento,Codigo_produto,Codigo_categoria,Marca,preco,Codigo_cliente
0,2019-10-01,view,44600062,2103807459595387724,shiseido,35.79,541312140
1,2019-10-01,view,3900821,2053013552326770905,aqua,33.20,554748717
2,2019-10-01,view,17200506,2053013559792632471,Sem marcas,543.10,519107250
3,2019-10-01,view,1307067,2053013558920217191,lenovo,251.74,550050854
4,2019-10-01,view,1004237,2053013555631882655,apple,1081.98,535871217


In [76]:
## Criacao de views com principais analises clientes produtos e dados de tempos

con.execute("""Select 
                Tipo_evento, 
                YEAR(Tempo_evento) AS ANO,
                month(Tempo_evento) AS mes,
                count(Tipo_evento) as contar 
            from vendas_limpas 
            group by Tipo_evento, YEAR(Tempo_evento), month(Tempo_evento)""").fetch_df()


,Tipo_evento,ANO,mes,contar
0,view,2019,10,40779399
1,purchase,2019,10,742849
2,cart,2019,10,926516


#### Exportar dados Para formato Parquet Particionado

#### Criar View analises frequentes e Modelagem Estrela 